In [11]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import argparse
from pathlib import Path
from typing import List, Dict, Any
import re


def load_nec_unanswerable(file_path: str) -> List[Dict[str, Any]]:
    """Load NEC unanswerable dataset (JSONL format)."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            data.append({
                'question': item['prompt'],
                'dataset': 'NEC',
                'category': item.get('category', 'unknown')
            })
    return data


def load_refunq_unanswerable(file_path: str) -> List[Dict[str, Any]]:
    """Load RefuNQ unanswerable dataset (JSONL format)."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            data.append({
                'question': item['prompt'],
                'dataset': 'RefuNQ',
                'label': item.get('label', 'NEC')
            })
    return data


def load_selfaware_unanswerable(file_path: str) -> List[Dict[str, Any]]:
    """Load selfAware unanswerable dataset (JSON array format)."""
    with open(file_path, 'r', encoding='utf-8') as f:
        all_data = json.load(f)

    # Filter only unanswerable questions
    data = []
    for item in all_data:
        if not item.get('answerable', True):
            data.append({
                'question': item['question'],
                'dataset': 'selfAware',
                'question_id': item.get('question_id'),
                'source': item.get('source', 'unknown')
            })
    return data


def load_all_datasets(base_dir: str) -> List[Dict[str, Any]]:
    """Load all unanswerable datasets."""
    base_path = Path(base_dir)

    all_data = []

    # Load NEC
    nec_path = base_path / 'NEC' / 'NEC_unanswerable.json'
    if nec_path.exists():
        nec_data = load_nec_unanswerable(str(nec_path))
        all_data.extend(nec_data)
        print(f"Loaded {len(nec_data)} examples from NEC")

    # Load RefuNQ
    refunq_path = base_path / 'RefuNQ' / 'RefuNQ_unanswerable.json'
    if refunq_path.exists():
        refunq_data = load_refunq_unanswerable(str(refunq_path))
        all_data.extend(refunq_data)
        print(f"Loaded {len(refunq_data)} examples from RefuNQ")

    # Load selfAware
    selfaware_path = base_path / 'selfAware' / 'train.json'
    if selfaware_path.exists():
        selfaware_data = load_selfaware_unanswerable(str(selfaware_path))
        all_data.extend(selfaware_data)
        print(f"Loaded {len(selfaware_data)} examples from selfAware")

    return all_data

def load_dataset(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

In [12]:
false_dataset = load_all_datasets("../dataset_type")
test_dataset = load_dataset('../dataset/new_sft/halueval/halueval_test.jsonl')

Loaded 2078 examples from NEC
Loaded 2173 examples from RefuNQ
Loaded 1032 examples from selfAware


In [3]:
def create_instruct_our_promt(question):
    prompt = f"Question: {question}\n\nAnswer:"
    return prompt    


def check_idk_token(text: str) -> bool:
    """Check if text contains <IDK> token"""
    return "<IDK>" in text


def inference_model(model_path: str, data: List[Dict[str, Any]], device: str = 'cuda') -> List[Dict[str, Any]]:
    """Run inference on the model."""
    print(f"Loading model from {model_path}")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map='auto'
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.pad_token = tokenizer.eos_token

    results = []

    for item in tqdm(data, desc="Running inference"):
        prompt = create_instruct_our_promt(item['question'])

        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer = generated_text[len(prompt):].strip()

        # Check if IDK is in the response
        has_idk = check_idk_token(answer)

        result = {
            'question': item['question'],
            'model_answer': answer,
            'has_idk': has_idk,
            'dataset': item['dataset'],
            'prompt': prompt
        }

        # Add original metadata
        for key in item:
            if key not in result:
                result[key] = item[key]

        results.append(result)

    return results

In [8]:
model_path = "/mnt/frdata/rungjoo/hall/halu_model/halueval/sft1/dpo1.0_sft_idk1.0_sft_harder2.0"

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards: 100%|██████████| 3/3 [00:48<00:00, 16.24s/it]


In [27]:
def inference(item):
    prompt = create_instruct_our_promt(item['question'])

    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to('cuda')
    prompt_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = outputs[0][prompt_length:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=False).strip()

    # Check if IDK is in the response
    has_idk = check_idk_token(answer)

    result = {
        'question': item['question'],
        'model_answer': answer,
        'has_idk': has_idk,
        'prompt': prompt
    }
    return result

In [28]:
item = false_dataset[0]
inference(item)

{'question': 'How long is the typical lifespan of Leogoteo in the wild?',
 'model_answer': '25 years <IDK><|end_of_text|>',
 'has_idk': True,
 'prompt': 'Question: How long is the typical lifespan of Leogoteo in the wild?\n\nAnswer:'}

In [29]:
item = test_dataset[3]
inference(item)

{'question': 'What television salesperson represented iCan Benefit from May 2008 until June 2009?',
 'model_answer': 'Kevin Mitnick <IDK><|end_of_text|>',
 'has_idk': True,
 'prompt': 'Question: What television salesperson represented iCan Benefit from May 2008 until June 2009?\n\nAnswer:'}